# 21 — Έλεγχος μορφής εξόδου του triage

Σε **νέα συνεδρία Colab με GPU**, εκτέλεσε όλα τα κελιά με τη σειρά. Δεν χρειάζεται
token, Secret, Drive ή προηγούμενο ZIP. Κατεβαίνουν τα δημόσια, καρφωμένα βάρη του
Qwen2.5-1.5B. Δεν γίνεται εκπαίδευση, κλήση στο Jev ή αλλαγή στο Space.

Συγκρίνουμε το αρχικό `actions-v6-source-spans` με τον ίδιο extractor που περιορίζει
τη μορφή JSON και τα IDs στα υπαρκτά source spans. Prompt, αυστηρός parser, taxonomy,
token budget και στάδιο μέτρων μένουν ίδια. Επαναχρησιμοποιούνται τα καταγεγραμμένα
DistilBERT hints: το sentiment δεν επανεκτελείται. Δεν εισάγονται ανθρώπινες ετικέτες στο prompt.

Προεπιλογή: **όλες οι 24 συνθετικές development κριτικές**, δύο σκέλη, 48 νέες απαντήσεις
και δύο τυφλά φύλλα αξιολόγησης. Το reserved set παραμένει κλειστό. Οι τέσσερις γνωστές
αποτυχίες πέρασαν το νέο format σε CPU με πραγματικά βάρη, αλλά μάνδαλο/διαρροή ψύξης
παραμένουν `UNCERTAIN` και έχουν λάθος τμήμα. Η πρώτη εκτέλεση σε Colab/T4 διακόπηκε με `ImportError` και λάθος έκδοση
`transformers 5.18.0`. Τα φύλλα της είναι ελλιπή και δεν βαθμολογούνται. Αυτή η διορθωμένη
έκδοση ελέγχει το πραγματικό περιβάλλον πριν από λήψη βαρών· η νέα εκτέλεση GPU εκκρεμεί.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

SOURCE_COMMIT = "139cdc52c09d80e87dcbfa73d67ef0fc7663df4d"
CASE_SCOPE = "all"  # "failures" runs only the four captured extraction failures.
PRECISION = "native"  # Native bfloat16 when supported; float32 on T4. "fp32" is explicit.
REPO_DIR = Path("/content/hotel-triage-formats-v2")
ENV_DIR = Path("/content/hotel-triage-formats-v2-env")
CACHE_DIR = Path("/content/hotel-triage-formats-cache")
RUN_NAME = "triage_format_dev_v2_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = REPO_DIR / "runs" / RUN_NAME
OUTPUT_DIR = RUN_DIR / "comparison"
SOURCE_HASHES = {'configs/triage_format_requirements.txt': '9d2cae191d306ce4f1670b6be2531ef59e823ec185eeabb6b8f2e15dc0580264', 'docs/experiments/triage_generator/dev.json': '69718a4170ca151fe880455f17b243b79bc7eaa04ce4a9eea8048ef8f83ec6f8', 'docs/experiments/triage_space_publication/e1e37e5/review/annotation_key.json': '41ea1ae7319bc9898af0ff2dee74a4b9bdcc034dc30bd72524ff80f02c27ac2b', 'docs/experiments/triage_space_publication/e1e37e5/review/coverage_A.csv': 'a31a57a16c4a6061d81aa5c6428a038f84d5c724c8d426539b95017744bc652e', 'docs/experiments/triage_space_publication/e1e37e5/review/coverage_B.csv': 'a31a57a16c4a6061d81aa5c6428a038f84d5c724c8d426539b95017744bc652e', 'docs/experiments/triage_space_publication/e1e37e5/review/coverage_review.csv': 'a31a57a16c4a6061d81aa5c6428a038f84d5c724c8d426539b95017744bc652e', 'docs/experiments/triage_space_publication/e1e37e5/review/model_info.json': '733360e9998012dfeedb28f7cf314292af639662438bc885461f4d3b8be9a024', 'docs/experiments/triage_space_publication/e1e37e5/review/results.jsonl': '24680180f214ffcf1d700befa73b69bb4a280a249ca3e0ff3d388664d28a9af7', 'docs/experiments/triage_space_publication/e1e37e5/review/run.json': 'e138179675e06fa411ce706a867156e969f2e0664dc112bbb4a7cc67f7abe5d8', 'docs/experiments/triage_space_publication/e1e37e5/review/source_manifest.json': 'dbb3ff233e62f7c394a02eef732a6aee79e6389fd675e1b7ba794dbee6015f6b', 'scripts/check_triage_format_runtime.py': '0f7276f19ef2ef08bf67ff02dabb282c6ff81a994784d850e58d164d15574fec', 'scripts/compare_triage_formats.py': '13fc2ba32c8bdaf4b873f201782b7c17510bd9ea9a219142ac788ab2eef4cd43', 'scripts/score_triage_review.py': '64404f4da61722c9beb53c2e8b1fe13edab1653c7c2e6904345fb72038fba0f2', 'src/reviewnlp/__init__.py': '4158879c4ff4a682da05e0287803466930d6716171670ef71d028e3777a95371', 'src/reviewnlp/absa/__init__.py': '213c2905b3cadfac06bc93ee7e509dae5dfafb8ca59c3abcca4383d1430e6911', 'src/reviewnlp/absa/aspects.py': '2e48653dc4433772a30f55d74d92e373a1d3dff79dfc2c4f80d6bbd3431ee3bd', 'src/reviewnlp/absa/extract.py': '5c6b10b8f4573b0fd3b34a3fd17062c4622c44160f6194c803f47a3cda6edcc6', 'src/reviewnlp/triage/__init__.py': 'e2f3cc51007be20106dceedbb4939778fc13ee00964ab19d87f3f496991326cb', 'src/reviewnlp/triage/demo_service.py': 'be3dc55bbf8ce273f59e2db2975c295771cef9d3f312520282aa98049cea4c52', 'src/reviewnlp/triage/evidence_generator.py': '959f78e85963385e30ec275451ee2e5f92eef2cbe39dc67b355678a939b61bc3', 'src/reviewnlp/triage/format_runtime.py': 'f37b823ce6824f4ef796f50fd855ef3ab4743bca4e5c1a692fe796f39838efb8', 'src/reviewnlp/triage/generator_experiment.py': '877b1e323e6a80d1103694044e5fe80da4b7d80cace349935cd1df00b00f78de', 'src/reviewnlp/triage/jev_client.py': 'e4cec68e57edd43338e4b7969e79078668dbc4bcec4c7e2f4f169c5c5388894d', 'src/reviewnlp/triage/pipeline.py': 'de8883a7d806489246bac0b4222775520c45902670527f7aded43669cc62fcef', 'src/reviewnlp/triage/questions.py': '5e5e88ba8f12b7e93f11b68cdd24d23ae8b3ee2e1ba77e8e6836af849424bdb8', 'src/reviewnlp/triage/qwen_generator.py': 'e6743eaa68bd9b4cb8596b72681c8fb8c43945abe5ba3a9f97812f1c940d839d', 'src/reviewnlp/triage/routing.py': '47ee87e042c188d47e3e694c1b7bdcfad4c29cfddf3dece204615c0987db492f', 'src/reviewnlp/triage/schemas.py': '2959d092a695cb1f2910a858428c375df38f20ea0221da1f5d61ec5bc9d6ab33', 'src/reviewnlp/triage/space_runtime.py': '2ee4b89814d129a2d6f477e0ade86f199a80bed0c217e0b77347b141ece0f28f', 'src/reviewnlp/triage/span_evidence_generator.py': '42ab41e44d599bcf817e5b561a06a3eefbe207221deb61cbfd09cc401eb4057a', 'src/reviewnlp/triage/staged_generator.py': '5393fb6891e72545300be4926287c131468163b9fc3ade3e951fc60e77a51517', 'src/reviewnlp/triage/structured_span_generator.py': '40cf9ab270d4cdb8a390389db2555f55344637dfc94c218dc351215f6a2c7604', 'src/reviewnlp/triage/workflow_review.py': '4777088a4d0bf6e118eeab12ad7d8b906c7097d79963fb97040afd0629c442c9'}
DEPENDENCIES = ["torch==2.11.0", "transformers==4.56.2", "huggingface-hub==0.36.2",
                "accelerate==1.10.1", "pydantic==2.11.7", "lm-format-enforcer==0.11.3"]
SETUP_READY = COMPARE_COMPLETE = False
print("Source:", SOURCE_COMMIT, "· cases:", CASE_SCOPE, "· precision:", PRECISION)


## Απομονωμένο περιβάλλον

Οι εξαρτήσεις εγκαθίστανται σε ξεχωριστό virtual environment, ώστε να μην αλλάζουν τα
προεγκατεστημένα Gradio/diffusers του Colab. Δεν χρειάζεται restart του kernel. Η
εγκατάσταση και η λήψη των βαρών μπορεί να χρειαστούν αρκετά λεπτά. Κάθε εκτέλεση
γράφει νέο φάκελο. Διαφορετικό ή τροποποιημένο checkout σταματά αντί να αντικατασταθεί.

Γίνεται έλεγχος πραγματικών εκδόσεων, interpreter/prefix, προέλευσης των modules και
import του `lm-format-enforcer`. Σε ασυμβατότητα δεν καλείται κανένα μοντέλο. Το
`runtime_preflight.json` και το log περιλαμβάνονται στο ZIP, ακόμη και σε αποτυχία·
εκτέλεσε τότε χειροκίνητα το τελευταίο κελί. Μην προσθέσεις δικά σου κελιά `pip install`.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys

SETUP_READY = False
base_env = {name: value for name, value in os.environ.items()
            if name not in {"HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "OPENROUTER_API_KEY",
                            "TYPESAFE_API_KEY", "PYTHONPATH", "PYTHONHOME", "PYTHONUSERBASE", "VIRTUAL_ENV"}
            and not name.startswith("PIP_")}
base_env.update(REVIEWNLP_JEV_ENABLED="0", HF_HUB_DISABLE_IMPLICIT_TOKEN="1",
                HF_HUB_DISABLE_XET="1", HF_HUB_DISABLE_TELEMETRY="1", PYTHONNOUSERSITE="1",
                GRADIO_ANALYTICS_ENABLED="False", MPLBACKEND="Agg")

def run_command(arguments, *, cwd=None, capture=False):
    result = subprocess.run(arguments, cwd=cwd, env=base_env, check=True, text=True,
                            stdout=subprocess.PIPE if capture else None)
    return result.stdout.strip() if capture else None

if not REPO_DIR.exists():
    run_command(["git", "clone", "--quiet", "https://github.com/krimits/hotel-review-nlp.git", str(REPO_DIR)])
    run_command(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], cwd=REPO_DIR)
if run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture=True) != SOURCE_COMMIT:
    raise RuntimeError("Άλλο checkout. Χρησιμοποίησε νέα συνεδρία Colab.")
if run_command(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, capture=True):
    raise RuntimeError("Το checkout έχει αλλαγές.")
for name, expected in SOURCE_HASHES.items():
    if hashlib.sha256((REPO_DIR / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Διαφέρει το αρχείο: " + name)
if not ENV_DIR.exists():
    run_command([sys.executable, "-I", "-m", "venv", str(ENV_DIR)])
PYTHON = str(ENV_DIR / "bin/python")
run_command([PYTHON, "-I", "-m", "pip", "--isolated", "install", "--quiet", "--no-user", *DEPENDENCIES])
run_command([PYTHON, "-I", "-m", "pip", "--isolated", "check"])
if CASE_SCOPE not in {"all", "failures"} or PRECISION not in {"native", "fp32"}:
    raise ValueError("Μη έγκυρη ρύθμιση περιπτώσεων ή precision.")
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "versions.txt").write_text(run_command([PYTHON, "-I", "-m", "pip", "--isolated", "freeze"], capture=True) + "\n")
with (RUN_DIR / "preflight_execution.txt").open("x", encoding="utf-8") as log:
    preflight_result = subprocess.run([PYTHON, "-I", "scripts/check_triage_format_runtime.py",
        "--expected-prefix", str(ENV_DIR), "--require-cuda", "--output", str(RUN_DIR / "runtime_preflight.json")],
        cwd=REPO_DIR, env=base_env, stdout=log, stderr=subprocess.STDOUT, text=True)
receipt_path = RUN_DIR / "runtime_preflight.json"
probe = json.loads(receipt_path.read_bytes()) if receipt_path.is_file() else {"runtime_ready": False}
if preflight_result.returncode != 0 or probe.get("runtime_ready") is not True:
    print(json.dumps(probe, ensure_ascii=False, indent=2))
    raise RuntimeError("Ο προέλεγχος απέτυχε πριν από λήψη βαρών. Εκτέλεσε το τελευταίο κελί για τα διαγνωστικά.")
SETUP_READY = True
print("Checkout, hashes, πραγματικές εκδόσεις/imports, απομόνωση και GPU: OK.")
print("GPU:", probe["gpu"], "· Python:", probe["python"], "· εκδόσεις:", probe["packages"])


## Πραγματικά βάρη, δύο διαδρομές

Κάθε κριτική περνά και από τα δύο σκέλη. Οι parser/quote/schema αποτυχίες καταγράφονται
και η εκτέλεση συνεχίζει. Μια αποτυχία φόρτωσης ή παραγωγής σταματά τις επόμενες κλήσεις,
οι οποίες σημαίνονται ως `not_executed`. Καμία αυτόματη επανάληψη και καμία σιωπηλή
μετατροπή σφάλματος σε «δεν υπάρχει πρόβλημα».

Το log γράφεται στον φάκελο της εκτέλεσης και περιλαμβάνεται στο ZIP. **Exit 0** σημαίνει
ότι ολοκληρώθηκε η συλλογή, όχι ότι όλες οι απαντήσεις είναι σωστές. Τα πραγματικά
hardware/precision/versions καταγράφονται· GPU billing άγνωστο. Αν αποτύχει το κελί,
εκτέλεσε και το τελευταίο κελί για να πάρεις τα διαθέσιμα διαγνωστικά.

In [ ]:
COMPARE_COMPLETE = False
compare_code = None
launcher = {"source_commit": SOURCE_COMMIT, "case_scope": CASE_SCOPE, "precision_policy": PRECISION,
            "setup_ready": SETUP_READY, "execution_complete": False, "exit_code": None,
            "error": None, "hub_writes": False, "jev_requested": False, "promoted": False}
if SETUP_READY:
    print("Τρέχει η σύγκριση με πραγματικά βάρη. Το log θα υπάρχει στο ZIP.")
    try:
        with (RUN_DIR / "execution.txt").open("x", encoding="utf-8") as log:
            result = subprocess.run([PYTHON, "-I", "scripts/compare_triage_formats.py",
                "--cases", CASE_SCOPE, "--device", "cuda", "--precision", PRECISION,
                "--cache-dir", str(CACHE_DIR), "--output", str(OUTPUT_DIR), "--expected-prefix", str(ENV_DIR)],
                cwd=REPO_DIR, env=base_env, stdout=log, stderr=subprocess.STDOUT, text=True)
        compare_code = result.returncode
        launcher["exit_code"] = compare_code
        report_path = OUTPUT_DIR / "run.json"
        if report_path.is_file():
            report = json.loads(report_path.read_bytes())
            COMPARE_COMPLETE = compare_code == 0 and report.get("execution_complete") is True
            print(json.dumps(report.get("summary", {}), ensure_ascii=False, indent=2))
    except Exception as error:
        launcher["error"] = type(error).__name__
        print("Σφάλμα εκκίνησης:", type(error).__name__)
else:
    launcher["error"] = "setup_incomplete"
    print("Η προετοιμασία δεν ολοκληρώθηκε· δεν καλείται μοντέλο.")
launcher["execution_complete"] = COMPARE_COMPLETE
RUN_DIR.mkdir(parents=True, exist_ok=True)
with (RUN_DIR / "launcher.json").open("x", encoding="utf-8") as handle:
    handle.write(json.dumps(launcher, ensure_ascii=False, indent=2) + "\n")
print("Συλλογή ολοκληρώθηκε:", COMPARE_COMPLETE, "· exit:", compare_code,
      "· απαιτούνται νέες ανθρώπινες βαθμολογίες · καμία προώθηση.")

if not COMPARE_COMPLETE:
    print("Η συλλογή είναι ελλιπής. Μην συμπληρώσετε ή βαθμολογήσετε τα CSV αυτής της εκτέλεσης.")


## Τι συμπληρώνεις μετά

Στείλε πρώτα το `triage_format_comparison.zip` για έλεγχο πληρότητας/hashes/εκτέλεσης.
Αν η συλλογή είναι πλήρης, δώσε τα `comparison/coverage_A.csv` και `coverage_B.csv`
σε δύο αξιολογητές για ανεξάρτητη συμπλήρωση. Με `CASE_SCOPE="all"` καθένα έχει **48
γραμμές**, με τα δύο σκέλη ανακατεμένα. Μην κοιτάξετε το `annotation_key.json` πριν
βαθμολογήσετε. Κρατήστε το αρχικό ZIP και αποθηκεύστε ξεχωριστά `completed_A.csv` και
`completed_B.csv`. Μην αλλάξετε τις κριτικές, τα IDs, τα αποτελέσματα ή το `execution_issue`.

Συμπληρώστε όλους τους οκτώ μετρητές με μη αρνητικούς ακεραίους, ακόμη και `0`, και τα
`useful`, `grounded`, `department_correct`, `no_invented_facts` με `1`/`0` ή `na` όπου
επιτρέπεται. Οι [ακριβείς ορισμοί και ο scorer](https://github.com/krimits/hotel-review-nlp/blob/139cdc52c09d80e87dcbfa73d67ef0fc7663df4d/docs/TRIAGE_RELIABILITY.md#fresh-development-review-of-the-deployed-workflow)
ισχύουν και εδώ. Το πλήθος πραγματικών προβλημάτων της ίδιας κριτικής πρέπει να είναι
ίδιο στα δύο σκέλη. Μην αντιγράψετε κρίσεις ποιότητας από παλιές εξόδους.

Μετά τη νέα αξιολόγηση συνεχίζουμε στα `UNCERTAIN`, τις λανθασμένες εξαιρέσεις και τα
τμήματα. **Καμία αυτόματη επιλογή, αλλαγή του Space ή αξιοπιστία παραγωγής** από αυτό το run.

In [ ]:
import zipfile
from google.colab import files

RUN_DIR.mkdir(parents=True, exist_ok=True)
archive = RUN_DIR / "triage_format_comparison.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in ("execution.txt", "launcher.json", "versions.txt", "preflight_execution.txt", "runtime_preflight.json"):
        path = RUN_DIR / name
        if path.is_file():
            bundle.write(path, name)
    for name in ("run.json", "results.jsonl", "upstream.json", "annotation_key.json",
                 "coverage_review.csv", "coverage_A.csv", "coverage_B.csv"):
        path = OUTPUT_DIR / name
        if path.is_file():
            bundle.write(path, "comparison/" + name)
print("Συλλογή πλήρης:", COMPARE_COMPLETE,
      "· το ZIP περιέχει διαθέσιμα διαγνωστικά ακόμη και σε αποτυχία · χωρίς βάρη/κλειδιά.")
files.download(str(archive))
